In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import os

DATA_PATH = "/Users/natallia/MIS/Lab1/"
FILE_RATINGS = "u.data"
FILE_MOVIES = "u.item"

In [13]:

try:
    ratings_filepath = os.path.join(DATA_PATH, FILE_RATINGS)
    ratings = pd.read_csv(ratings_filepath, sep='\t',
                          names=["user_id", "movie_id", "rating", "unix_timestamp"])

    movies_filepath = os.path.join(DATA_PATH, FILE_MOVIES)
    movies = pd.read_csv(movies_filepath, sep="|",
                          usecols=[0, 1],
                          names=["movie_id", "title"],
                          encoding="latin-1")
    
except FileNotFoundError as e:
    print(f"Ошибка: Файл не найден. Проверьте путь и имя файла: {e}")
    exit()

In [17]:
print("\n Создание матрицы 'Пользователь-Фильм'")

user_movie_matrix = ratings.pivot_table(index='user_id', columns='movie_id', values='rating')

print(f"\n Матрица 'Пользователь-Фильм' создана. Размерность: {user_movie_matrix.shape}")


 Создание матрицы 'Пользователь-Фильм'

 Матрица 'Пользователь-Фильм' создана. Размерность: (943, 1682)


In [21]:

user_movie_filled = user_movie_matrix.fillna(0) 
movie_user_filled = user_movie_filled.T


movie_user_sparse_matrix = csr_matrix(movie_user_filled.values) 


item_similarity_matrix = cosine_similarity(movie_user_sparse_matrix)
print("Матрица вычислена.")


item_similarity_df = pd.DataFrame(item_similarity_matrix, 
                                  index=movie_user_filled.index, 
                                  columns=movie_user_filled.index)

id_to_title = movies.set_index('movie_id')['title'].to_dict()

print(item_similarity_df)

Матрица вычислена.
movie_id      1         2         3         4         5         6     \
movie_id                                                               
1         1.000000  0.402382  0.330245  0.454938  0.286714  0.116344   
2         0.402382  1.000000  0.273069  0.502571  0.318836  0.083563   
3         0.330245  0.273069  1.000000  0.324866  0.212957  0.106722   
4         0.454938  0.502571  0.324866  1.000000  0.334239  0.090308   
5         0.286714  0.318836  0.212957  0.334239  1.000000  0.037299   
...            ...       ...       ...       ...       ...       ...   
1678      0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
1679      0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
1680      0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
1681      0.047183  0.078299  0.000000  0.056413  0.000000  0.000000   
1682      0.047183  0.078299  0.096875  0.075218  0.094211  0.000000   

movie_id      7         8         9         

In [19]:
def recommend_collaborative(title, item_sim_df=item_similarity_df, id_to_title_map=id_to_title):
    
    print("\n" + "=" * 50)
    print(f"Поиск рекомендаций для: '{title}'")
    
    movie_data = movies[movies.title == title]
    if movie_data.empty:
        print(f"Ошибка: Фильм '{title}' не найден в списке.")
        return

    movie_id = movie_data['movie_id'].values[0]

    if movie_id not in item_sim_df.index:
        print(f"Ошибка: Фильм '{title}' (ID: {movie_id}) не имеет оценок в данных для CF.")
        return

    similarity_series = item_sim_df.loc[movie_id]
    
    top_similar_movies = similarity_series.sort_values(ascending=False).iloc[1:6]

    print("Топ-5 рекомендаций по схожести оценок пользователей:")
    for rec_movie_id, score in top_similar_movies.items():
        rec_title = id_to_title_map.get(rec_movie_id, f"ID {rec_movie_id} (название не найдено)")
        print(f"- {rec_title}: Сходство {score:.4f}")

In [20]:
recommend_collaborative('Star Wars (1977)')
recommend_collaborative('Titanic (1997)')


Поиск рекомендаций для: 'Star Wars (1977)'
Топ-5 рекомендаций по схожести оценок пользователей:
- Return of the Jedi (1983): Сходство 0.8845
- Raiders of the Lost Ark (1981): Сходство 0.7649
- Empire Strikes Back, The (1980): Сходство 0.7498
- Toy Story (1995): Сходство 0.7346
- Godfather, The (1972): Сходство 0.6973

Поиск рекомендаций для: 'Titanic (1997)'
Топ-5 рекомендаций по схожести оценок пользователей:
- Good Will Hunting (1997): Сходство 0.5912
- Contact (1997): Сходство 0.5530
- Apt Pupil (1998): Сходство 0.5484
- Tomorrow Never Dies (1997): Сходство 0.5345
- Air Force One (1997): Сходство 0.5304
